In [3]:
import pandas as pd
import numpy as np
from scipy.stats import poisson


In [6]:
#Loading the csv file 
df = pd.read_csv('data/premier_league_stats.csv')


In [7]:
df

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,8,6,6,4,21,20,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9375,2024/25,2025-05-04,Brentford,Man United,4,3,H,2,1,H,...,6,5,7,4,8,10,0,2,0,0
9376,2024/25,2025-05-04,Brighton,Newcastle,1,1,D,1,0,H,...,2,5,1,4,15,10,2,1,0,0
9377,2024/25,2025-05-04,West Ham,Tottenham,1,1,D,1,1,D,...,2,2,1,3,18,15,2,2,0,0
9378,2024/25,2025-05-04,Chelsea,Liverpool,3,1,H,1,0,H,...,7,2,3,6,10,11,2,2,0,0


In [8]:
#  Group by HomeTeam and calculate the mean
avg_goals = df.groupby('HomeTeam')[['FullTimeHomeGoals']].mean().round(2)
avg_goals = avg_goals.reset_index()
#Sort the teams so the highest scorers are at the top
top_teams = avg_goals.sort_values(by='FullTimeHomeGoals', ascending=False)
#removing the messy index 
top_teams = top_teams.reset_index(drop=True)
# Add a rank column starting from 1
top_teams.insert(0, 'Rank', range(1, len(top_teams) + 1))
# Top 10 team 
print(top_teams.head(10).to_string(index=False))


 Rank   HomeTeam  FullTimeHomeGoals
    1   Man City               2.19
    2    Arsenal               2.15
    3  Liverpool               2.08
    4    Chelsea               2.06
    5 Man United               2.04
    6  Tottenham               1.84
    7  Brentford               1.65
    8  Blackpool               1.58
    9  Newcastle               1.57
   10    Everton               1.51


In [9]:
#  Group by AwTeam and calculate the mean
avg_goals = df.groupby('AwayTeam')[['FullTimeAwayGoals']].mean().round(2)
avg_goals = avg_goals.reset_index()
#Sort the teams so the highest scorers are at the top
top_teams = avg_goals.sort_values(by='FullTimeAwayGoals', ascending=False)
#removing the messy index 
top_teams = top_teams.reset_index(drop=True)
# Add a rank column starting from 1
top_teams.insert(0, 'Rank', range(1, len(top_teams) + 1))
# Top 10 team 
print(top_teams.head(10).to_string(index=False))

 Rank   AwayTeam  FullTimeAwayGoals
    1    Arsenal               1.65
    2  Liverpool               1.65
    3   Man City               1.63
    4 Man United               1.59
    5    Chelsea               1.57
    6  Tottenham               1.41
    7  Brentford               1.35
    8      Leeds               1.35
    9  Blackpool               1.32
   10  Leicester               1.31


In [10]:

# Let's rename the columns so they are easy to merge
home_data = df.groupby('HomeTeam')[['FullTimeHomeGoals', 'HomeShotsOnTarget']].mean().reset_index()
home_data.columns = ['Team', 'AvgHomeGoals', 'AvgHomeShotsOnTarget']

away_data = df.groupby('AwayTeam')[['FullTimeAwayGoals', 'AwayShotsOnTarget']].mean().reset_index()
away_data.columns = ['Team', 'AvgAwayGoals', 'AvgAwayShotsOnTarget']

master_df = pd.merge(home_data, away_data, on='Team')

# Home Advantage = AvgHomeGoals - AvgAwayGoals
master_df['HomeAdvantage'] = (master_df['AvgHomeGoals'] - master_df['AvgAwayGoals']).round(2)

# Conversion rates
master_df['HomeConversionRate'] = (master_df['AvgHomeGoals'] / master_df['AvgHomeShotsOnTarget']).round(2)
master_df['AwayConversionRate'] = (master_df['AvgAwayGoals'] / master_df['AvgAwayShotsOnTarget']).round(2)
master_df['OverallConversionRate'] = ((master_df['AvgHomeGoals'] + master_df['AvgAwayGoals']) / (master_df['AvgHomeShotsOnTarget'] + master_df['AvgAwayShotsOnTarget'])).round(2)

# Top 5 Home Dependent (High HomeAdvantage)
home_dependent = master_df.sort_values(by='HomeAdvantage', ascending=False).head(5).reset_index(drop=True)
home_dependent.insert(0, 'Rank', range(1, len(home_dependent) + 1))
home_dependent['HomeAdvantage'] = home_dependent['HomeAdvantage'].round(2)
home_dependent = home_dependent[['Rank', 'Team', 'HomeAdvantage']]

# Top 5 Road Warriors (High AvgAwayGoals)
road_warriors = master_df.sort_values(by='AvgAwayGoals', ascending=False).head(5).reset_index(drop=True)
road_warriors.insert(0, 'Rank', range(1, len(road_warriors) + 1))
road_warriors['AvgAwayGoals'] = road_warriors['AvgAwayGoals'].round(2)
road_warriors = road_warriors[['Rank', 'Team', 'AvgAwayGoals']]

# Correlation
correlation = round(df[['HomeShotsOnTarget', 'FullTimeHomeGoals']].corr().iloc[0, 1], 2)

# Teams with high conversion rates (overperforming)
overperformers = master_df.sort_values(by='OverallConversionRate', ascending=False).head(5).reset_index(drop=True)
overperformers.insert(0, 'Rank', range(1, len(overperformers) + 1))
overperformers['OverallConversionRate'] = overperformers['OverallConversionRate'].round(2)
overperformers = overperformers[['Rank', 'Team', 'OverallConversionRate']]

print('Home Dependent Teams:')
print(home_dependent.to_string(index=False))
print('\nRoad Warriors (High Avg Away Goals):')
print(road_warriors.to_string(index=False))
print(f'\nCorrelation between HomeShotsOnTarget and FullTimeHomeGoals: {correlation:.2f}')
print('\nOverperformers (High Conversion Rate):')
print(overperformers.to_string(index=False))

Home Dependent Teams:
 Rank       Team  HomeAdvantage
    1   Man City           0.55
    2   Bradford           0.53
    3      Stoke           0.52
    4 Portsmouth           0.51
    5    Arsenal           0.50

Road Warriors (High Avg Away Goals):
 Rank       Team  AvgAwayGoals
    1    Arsenal          1.65
    2  Liverpool          1.65
    3   Man City          1.63
    4 Man United          1.59
    5    Chelsea          1.57

Correlation between HomeShotsOnTarget and FullTimeHomeGoals: 0.44

Overperformers (High Conversion Rate):
 Rank          Team  OverallConversionRate
    1         Luton                   0.37
    2     Brentford                   0.34
    3 Nott'm Forest                   0.34
    4     Leicester                   0.31
    5   Bournemouth                   0.31


In [11]:
#Calculate average goals scored and conceded for home and away
# This creates our 'strength' metrics
avg_home_scored = df['FullTimeHomeGoals'].mean()
avg_away_scored = df['FullTimeAwayGoals'].mean()

In [12]:
# Calculate Home Attack/Defense strength
home_attack = df.groupby('HomeTeam')['FullTimeHomeGoals'].mean() / avg_home_scored
home_defense = df.groupby('HomeTeam')['FullTimeAwayGoals'].mean() / avg_away_scored

In [13]:
# Calculate Away Attack/Defense strength
away_attack = df.groupby('AwayTeam')['FullTimeAwayGoals'].mean() / avg_away_scored
away_defense = df.groupby('AwayTeam')['FullTimeHomeGoals'].mean() / avg_home_scored

In [14]:
def predict_goals(home_team, away_team):
    
    # Calculate Expected Goals (xG) for the match
    # Expected Home Goals = Home_Attack * Away_Defense * League_Avg_Home_Goals
    home_xg = home_attack[home_team] * away_defense[away_team] * avg_home_scored
    
    # Expected Away Goals = Away_Attack * Home_Defense * League_Avg_Away_Goals
    away_xg = away_attack[away_team] * home_defense[home_team] * avg_away_scored
    
    return home_xg, away_xg

In [ ]:
def predict_match_outcomes(home_team, away_team, max_goals=6):
    home_xg, away_xg = predict_goals(home_team, away_team)

    home_win_prob = 0.0
    draw_prob = 0.0
    away_win_prob = 0.0

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            prob = poisson.pmf(home_goals, home_xg) * poisson.pmf(away_goals, away_xg)

            if home_goals > away_goals:
                home_win_prob += prob
            elif home_goals < away_goals:
                away_win_prob += prob
            else:
                draw_prob += prob

    results = pd.DataFrame({
        'Outcome': ['Home Win', 'Draw', 'Away Win'],
        'Probability': [home_win_prob, draw_prob, away_win_prob]
    })
    results['ProbabilityPct'] = (results['Probability'] * 100).round(2)
    return results

# Example usage
h_team = 'Man City'
a_team = 'Middlesbrough'

outcomes = predict_match_outcomes(h_team, a_team)
print(f"Predicted xG: {h_team} ({predict_goals(h_team, a_team)[0]:.2f}) vs {a_team} ({predict_goals(h_team, a_team)[1]:.2f})")
print('\nMatch Outcome Probabilities:')


print(outcomes[['Outcome', 'ProbabilityPct']].to_string(index=False))

Predicted xG: Man City (2.03) vs Middlesbrough (0.67)

Match Outcome Probabilities:
 Outcome  ProbabilityPct
Home Win           68.91
    Draw           19.44
Away Win           11.16
